# 02 — Dense retrieval index over LEDGAR

Milestone **m2**: encode the LEDGAR training clauses with `all-MiniLM-L6-v2`, normalise embeddings, and retrieve the top-k clauses by cosine similarity. This notebook builds the index and presents 10 searches for manual inspection. It does not require a GPU; sentence-transformers will select an available device.

The clause bank is the **train split only**. The five pasted-clause examples below come from validation, so they are not their own matches in the index. Similarity is a relevance suggestion, not legal advice; inspect the returned text rather than treating the score as a relevance judgment.

## 1. Load LEDGAR and prepare the training clause bank

In [5]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # If the notebook runs from notebooks/
sys.path.insert(0, str(PROJECT_ROOT))

In [6]:
from src.data import load_ledgar, to_pandas
from src.retrieval.dense import MODEL_NAME, build_dense_index
from src.utils import set_seed

set_seed(42)
dataset = load_ledgar()
train_df = to_pandas(dataset, "train")
validation_df = to_pandas(dataset, "validation")

print(f"Training clause bank: {len(train_df):,} clauses")
print(f"Validation clauses for query examples: {len(validation_df):,}")
print(f"Embedding model: {MODEL_NAME}")

ledgar/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.9MB            

ledgar/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

ledgar/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.31MB            

ledgar/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

ledgar/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.44MB            

ledgar/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Training clause bank: 60,000 clauses
Validation clauses for query examples: 10,000
Embedding model: sentence-transformers/all-MiniLM-L6-v2


## 2. Build or load the cached embedding index

The cached vectors are stored under `data/indexes/` (gitignored). The cache is reused only when both the model name and the ordered clause-bank contents match.

In [7]:
retriever = build_dense_index(train_df, model_name=MODEL_NAME, batch_size=64)
cache_status = "loaded from cache" if retriever.cache_hit else "encoded and cached"
print(f"Index {cache_status}: {retriever.embeddings.shape[0]:,} clauses × {retriever.embeddings.shape[1]} dimensions")

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/938 [00:00<?, ?it/s]

Index encoded and cached: 60,000 clauses × 384 dimensions


## 3. Inspect 10 searches by hand

Five are lawyer-style questions and five are clauses held out from the training bank. Read the query and retrieved clause text together. For each search, note whether the results answer the question or are genuinely similar to the source clause; a shared broad topic alone is not enough. In particular, check whether numeric and jurisdiction constraints are actually preserved. These are qualitative checks, not a formal relevance-judgment set.

In [8]:
lawyer_questions = [
    "Find clauses that allow either party to terminate the agreement for convenience with 30 days' written notice.",
    "Show clauses requiring payment of invoices within 30 days of receipt.",
    "Find agreements governed by New York law, excluding its conflict-of-law rules.",
    "Show clauses that cap liability at the fees paid in the preceding 12 months.",
    "Find confidentiality clauses that allow disclosure to affiliates and professional advisers.",
]

validation_examples = validation_df.sample(n=5, random_state=42)
queries = [(f"Lawyer question {i + 1}", query) for i, query in enumerate(lawyer_questions)]
queries += [
    (f"Validation clause idx={int(row.idx)} ({row.label_name})", row.text)
    for row in validation_examples.itertuples(index=False)
]

search_rows = []
for query_name, query_text in queries:
    for rank, result in enumerate(retriever.search(query_text, k=5), start=1):
        search_rows.append({
            "query": query_name,
            "rank": rank,
            "score": round(result["score"], 4),
            "idx": result["idx"],
            "label_name": result["label_name"],
            "text": result["text"],
        })

import pandas as pd
results_df = pd.DataFrame(search_rows)
for query_name in results_df["query"].drop_duplicates():
    print(f"\n{query_name}")
    display(results_df.loc[results_df["query"] == query_name, ["rank", "score", "label_name", "text"]].reset_index(drop=True))


Lawyer question 1


,rank,score,label_name,text
0,1,0.8147,Terminations,The Term of this Agreement may be terminated b...
1,2,0.7835,Terminations,You or the Company may terminate this Agreemen...
2,3,0.7283,Terms,The term of this Agreement shall be for a peri...
3,4,0.7235,Terminations,The Agreement may be terminated at any time by...
4,5,0.7219,Terms,This Agreement may be terminated by either par...



Lawyer question 2


,rank,score,label_name,text
0,1,0.6841,Payments,The Company will pay each invoice submitted he...
1,2,0.6708,Payments,All amounts due under this Section shall be pa...
2,3,0.6614,Payments,Customer shall pay invoices issued by Supplier...
3,4,0.6474,Payments,Recipient shall compensate Provider only for S...
4,5,0.6318,Payments,Manager will invoice Partnership from time to ...



Lawyer question 3


...

,rank,score,label_name,text
0,1,0.8924,Entire Agreements,"This Agreement is the final expression of, and..."
1,2,0.8602,Entire Agreements,"This Agreement, along with the Confidentiality..."
2,3,0.8588,Entire Agreements,"This Agreement, together with all Schedules an..."
3,4,0.8587,Entire Agreements,This Agreement constitutes the entire agreemen...
4,5,0.8567,Entire Agreements,Other than the Company’s Non-Disclosure and No...



Validation clause idx=4742 (Cooperation)


,rank,score,label_name,text
0,1,0.9587,Cooperation,To facilitate the orderly conduct of the Compa...
1,2,0.8270,Expenses,The Company agrees promptly to reimburse the E...
2,3,0.8204,Cooperation,"During the Term and thereafter, Executive shal..."
3,4,0.8196,Cooperation,Executive agrees to reasonably cooperate with ...
4,5,0.8182,Cooperation,Executive shall cooperate with the Company to ...



Validation clause idx=4521 (Insurances)


,rank,score,label_name,text
0,1,0.9111,Insurances,Borrower and its Significant Subsidiaries main...
1,2,0.8940,Insurances,"The Borrower will, and will cause each of its ..."
2,3,0.8848,Insurances,"Each The Borrower will, and will cause each of..."
3,4,0.8842,Insurances,The properties of the Borrower and each Subsid...
4,5,0.8840,Insurances,The Borrower and its Subsidiaries carry insura...


## 4. Record the manual check

Use the table above to record which queries returned useful examples and where retrieval missed the intent. Do not use cosine similarity as a substitute for human relevance judgments. The broader lawyer-query evaluation, encoder comparison, and BM25 hybrid belong in notebook 05.